In [17]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from cdt.data import AcyclicGraphGenerator
from causallearn.search.ConstraintBased.PC import pc
from causallearn.graph.GeneralGraph import GeneralGraph
from pathlib import Path
import os

In [18]:
class CustomAcyclicGraphGenerator(AcyclicGraphGenerator):
    """
    A custom generator that uses a user-defined adjacency matrix
    to specify the causal graph structure. This class overrides the
    init_dag part of the base AcyclicGraphGenerator class in cdt.
    """
    
    def __init__(self, adjacency_matrix, causal_mechanism, **kwargs):
        
        # Get the number of nodes from the adjacency matrix
        nodes = adjacency_matrix.shape[0]
        super().__init__(causal_mechanism, nodes=nodes, **kwargs)
        # Store the provided adjacency matrix
        self.custom_adjacency_matrix = adjacency_matrix.copy()
    
    def init_dag(self, verbose=False):
        # Use the adjacency matrix we want (and not a random one as in the existing AcyclicGraphGenerator)
        self.adjacency_matrix = self.custom_adjacency_matrix.copy()
        
        # Create networkx graph and verify it's acyclic
        self.g = nx.DiGraph(self.adjacency_matrix)
        if list(nx.simple_cycles(self.g)):
            raise ValueError("Error: provided adjacency matrix contains cycles.")

In [19]:
def generate_chain_adj_matrix(n_features):

    adj_matrix = np.zeros((n_features, n_features), dtype=int)
    for i in range(n_features - 1):
        adj_matrix[i, i+1] = 1
        
    return adj_matrix

chain_100_adj_matrix = generate_chain_adj_matrix(100)

# print(chain_100_adj_matrix[:5, :5])
# print(chain_100_adj_matrix)

In [20]:
def add_direct_effects(input_adj_matrix,fraction):
    for i in range(1, fraction//2 +1):
        input_adj_matrix[i-1, input_adj_matrix.shape[0]-i] = 1
    return input_adj_matrix

In [21]:
loop_100_adj_matrix = add_direct_effects(generate_chain_adj_matrix(100),20)
print(loop_100_adj_matrix)
for row in loop_100_adj_matrix:
    print(row)

[[0 1 0 ... 0 0 1]
 [0 0 1 ... 0 1 0]
 [0 0 0 ... 1 0 0]
 ...
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 0]]
[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
[0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0]
[0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
[0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [22]:
causal_mechanisms = {
    '1': 'linear',
    '2': 'polynomial',
    '3': 'gaussian_process',
    '4': 'sigmoid',
    '5': 'nn'
}

adj_matrices = {
    1: chain_100_adj_matrix,
    2: loop_100_adj_matrix
}

dag_types = {
    1: 'chain',
    2: 'loop'
}

In [23]:
for dag_number in [1,2]:
    for i in range(100):
        generator = CustomAcyclicGraphGenerator(
            adjacency_matrix=adj_matrices[dag_number],
            causal_mechanism=causal_mechanisms['1'],
            npoints=1000
        )

        data, graph = generator.generate()

        folder = os.path.join(Path('./').resolve(), f'{dag_types[dag_number]}100data')
        file_name = f"dag{dag_number}dataset{i+1}.csv"
        file_path = os.path.join(folder, file_name)
        data.to_csv(file_path, index=False)

In [24]:
# print('generated the 2x2=4 datasets.')
# print('this has been added just so i have something to commit.')
# print('hopefully the datasets will NOT be committed!')

In [25]:
print('now the 100x2=200 datasets have been generated BUT they should be ignored by git!!')

now the 100x2=200 datasets have been generated BUT they should be ignored by git!!
